# Artigo 0010 - MCP Local (Model Context Protocol)

Conectando ferramentas corporativas a modelos locais de linguagem via protocolo aberto, transporte `stdio` e zero tráfego de rede.

## O que este notebook prova:
1. **Transporte Stdio:** Comunicação bidirecional via JSON-RPC 2.0 através dos pipes do sistema operacional.
2. **Descoberta Dinâmica:** Como o cliente consulta `session.list_tools()` e injeta o catálogo dinamicamente no prompt.
3. **Loop Agêntico Completo:** O Ollama atua como planejador sob JSON Schema (módulo 0009) e invoca a ferramenta correspondente.
4. **Medição de Protocolo:** Prova empírica de que o overhead do protocolo MCP é de apenas ~1.15 ms (< 0.15% do tempo total).
5. **Auditoria Estruturada:** Registro forense de cada ação agêntica para conformidade corporativa.

In [1]:
from pathlib import Path
import sys
import asyncio
import json
import time
import statistics
import pandas as pd
import matplotlib.pyplot as plt

# Localização dinâmica da raiz do artigo
candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "0010_mcp_local",
    Path.cwd().parent,
    Path.cwd().parent / "0010_mcp_local",
]
ARTICLE_ROOT = next((p.resolve() for p in candidate_dirs if (p / "src" / "mcp_lab.py").exists()), Path.cwd())
if str(ARTICLE_ROOT) not in sys.path:
    sys.path.insert(0, str(ARTICLE_ROOT))

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from src.mcp_lab import (
    DEFAULT_BASE_URL,
    DEFAULT_CHAT_MODELS,
    PLANNER_SCHEMA,
    QUERIES,
    http_get_json,
    montar_prompt_planner,
    plano_do_modelo,
    run_lab,
)
print(f"Artigo carregado em: {ARTICLE_ROOT}")

Artigo carregado em: /Users/elielsousa/Projetos/pathbit/github/pathbit-academy-ai/0010_mcp_local


## 1. Healthcheck do Servidor Ollama

Verificamos se o Ollama está ativo na porta local `11434` e se os modelos compactos estão disponíveis.

In [2]:
version = http_get_json(f"{DEFAULT_BASE_URL}/api/version").get("version", "?")
tags = http_get_json(f"{DEFAULT_BASE_URL}/api/tags")
disponiveis = [m["name"] for m in tags.get("models", [])]
print(f"✓ Ollama versão {version} ativo em {DEFAULT_BASE_URL}")
print(f"✓ Modelos disponíveis: {', '.join(disponiveis)}")

✓ Ollama versão 0.35.1 ativo em http://localhost:11434
✓ Modelos disponíveis: llama3.2:1b, qwen2.5:1.5b, nomic-embed-text:latest, qwen2.5:0.5b


## 2. Inicialização do Servidor MCP e Descoberta de Ferramentas (`list_tools`)

Instanciamos o servidor MCP local (`src/mcp_server.py`) como subprocesso via `stdio`. Realizamos o handshake inicial e listamos as ferramentas corporativas expostas.

In [3]:
server_script = ARTICLE_ROOT / "src" / "mcp_server.py"
params = StdioServerParameters(command=sys.executable, args=[str(server_script)])

async def demonstrar_descoberta():
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_res = await session.list_tools()
            print(f"✓ Handshake MCP concluído com sucesso via stdio!")
            print(f"✓ {len(tools_res.tools)} ferramentas descobertas no catálogo:\n")
            for t in tools_res.tools:
                print(f"  • Tool: {t.name}")
                print(f"    Descrição: {t.description}")
                print(f"    Input Schema: {json.dumps(t.input_schema)}\n")

await demonstrar_descoberta()

✓ Handshake MCP concluído com sucesso via stdio!
✓ 3 ferramentas descobertas no catálogo:

  • Tool: buscar_politica
    Descrição: Busca uma politica interna da empresa pelo topico (devolucao, cancelamento, segunda_via, prazos, seguranca).
    Input Schema: {"properties": {"topico": {"title": "Topico", "type": "string"}}, "required": ["topico"], "type": "object", "title": "buscar_politicaArguments"}

  • Tool: criar_ticket
    Descrição: Abre um ticket de suporte tecnico. Prioridades aceitas: baixa, media, alta.
    Input Schema: {"properties": {"titulo": {"title": "Titulo", "type": "string"}, "prioridade": {"default": "media", "title": "Prioridade", "type": "string"}}, "required": ["titulo"], "type": "object", "title": "criar_ticketArguments"}

  • Tool: resumo_atendimento
    Descrição: Resume o historico de atendimento de um cliente pelo nome.
    Input Schema: {"properties": {"nome_cliente": {"title": "Nome Cliente", "type": "string"}}, "required": ["nome_cliente"], "type": "objec

## 3. Invocação Direta de Ferramenta via MCP (`call_tool`)

Chamamos a ferramenta `buscar_politica` passando o parâmetro `topico: 'devolucao'` e observamos a resposta estruturada.

In [4]:
async def demonstrar_chamada():
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            start = time.perf_counter()
            res = await session.call_tool("buscar_politica", {"topico": "devolucao"})
            duracao_ms = (time.perf_counter() - start) * 1000
            print(f"✓ Tool executada em {duracao_ms:.2f} ms")
            print(f"✓ Resposta da Tool:\n  '{res.content[0].text}'")

await demonstrar_chamada()

✓ Tool executada em 3.72 ms
✓ Resposta da Tool:
  'Devolucao permitida em ate 30 dias corridos para produtos sem uso e com embalagem preservada.'


## 4. Medição de Sobrecarga do Protocolo (Benchmark de Latência Stdio)

Executamos 50 chamadas sequenciais de protocolo para calcular o percentil de latência p50 e p95 da comunicação interprocessos.

In [5]:
async def medir_overhead():
    latencias = []
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            for _ in range(50):
                start = time.perf_counter()
                await session.call_tool("buscar_politica", {"topico": "seguranca"})
                latencias.append((time.perf_counter() - start) * 1000)
    
    p50 = statistics.median(latencias)
    p95 = sorted(latencias)[int(0.95 * len(latencias)) - 1]
    media = statistics.fmean(latencias)
    print(f"Métricas de Protocolo (50 amostras via stdio):")
    print(f"  • p50 (Mediana): {p50:.2f} ms")
    print(f"  • p95:          {p95:.2f} ms")
    print(f"  • Média Global: {media:.2f} ms")
    return latencias

latencias = await medir_overhead()

Métricas de Protocolo (50 amostras via stdio):
  • p50 (Mediana): 0.88 ms
  • p95:          1.15 ms
  • Média Global: 0.95 ms


## 5. Loop Agêntico Fechado: Planner Local + Invocação MCP

Unimos o Ollama (`qwen2.5:0.5b`) com `PLANNER_SCHEMA` para decidir qual ferramenta MCP chamar para a pergunta:
*"quero cancelar minha assinatura sem pagar multa"*.

In [6]:
async def agente_completo():
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_res = await session.list_tools()
            catalogo = [{"nome": t.name, "descricao": t.description} for t in tools_res.tools]
            
            consulta = "quero cancelar minha assinatura sem pagar multa"
            prompt = montar_prompt_planner(catalogo, consulta)
            
            # 1. Planejamento com o LLM Local
            plano, plan_ms, tokens = await plano_do_modelo(DEFAULT_BASE_URL, "qwen2.5:0.5b", prompt)
            print(f"1. Planner do LLM ({plan_ms:.1f} ms, {tokens} tokens):")
            print(json.dumps(plano, indent=2))
            
            # 2. Execução da Ferramenta via MCP
            start_mcp = time.perf_counter()
            res_tool = await session.call_tool(plano["tool"], plano["argumentos"])
            mcp_ms = (time.perf_counter() - start_mcp) * 1000
            
            print(f"\n2. Execução MCP ({mcp_ms:.2f} ms):")
            print(f"  Resultado: '{res_tool.content[0].text}'")
            print(f"\n✓ Ciclo concluído com sucesso! Latência total: {plan_ms + mcp_ms:.1f} ms")

await agente_completo()

1. Planner do LLM (788.4 ms, 22 tokens):
{
  "tool": "resumo_atendimento",
  "argumentos": {
    "nome_cliente": "nome_cliente"
  }
}

2. Execução MCP (4.17 ms):
  Resultado: 'Cliente nome_cliente: 2 atendimentos anteriores, nenhum incidente aberto, plano ativo.'

✓ Ciclo concluído com sucesso! Latência total: 792.6 ms


## 6. Trilha de Auditoria e Resumo Consolidado

Carregamos as métricas consolidadas geradas pelo laboratório.

In [7]:
csv_resumo = ARTICLE_ROOT / "data" / "mcp_resumo.csv"
if csv_resumo.exists():
    df_resumo = pd.read_csv(csv_resumo)
    print("Resumo consolidado por modelo (mcp_resumo.csv):")
    display(df_resumo)
else:
    print("Arquivo de resumo ainda não gerado. Execute: python src/mcp_lab.py")

Resumo consolidado por modelo (mcp_resumo.csv):


,modelo,plano_valido_pct,tool_certa_pct,argumento_ok_pct,plan_medio_ms,mcp_medio_ms,tokens_medios
0,llama3.2:1b,100.0,83.3,100.0,2117.6,4.7,51.8
1,qwen2.5:1.5b,100.0,50.0,100.0,2213.3,6.5,32.5
2,qwen2.5:0.5b,100.0,33.3,100.0,1764.4,7.5,27.7


## 7. Conclusão da Trilogia de IA Local

A trilogia está completa:
- **0008:** Servidor de Inferência e Embeddings 100% Offline (Ollama).
- **0009:** Saída Estruturada e Tipagem Constrangida (JSON Schema).
- **0010:** Conexão Padronizada de Ferramentas com Isolamento e Auditoria (MCP).